# Familiar: Blood Transfusion Pack Analysis — Complete Solution

This notebook contains fully worked answers, alternate implementations, effect-size calculations, assumption diagnostics, visualizations, a bootstrap / sensitivity simulation section, and audience-aware commentary.

Data files must sit in the same directory as the notebook (`familiar_lifespan.csv`, `familiar_iron.csv`).


## Flowchart: Desired Outcome for the Familiar Analysis

This flowchart captures the full recommended process for turning the two CSV files into actionable product insights for Familiar.  
It emphasises **exploratory checks → formal hypothesis tests → practical effect sizes → audience-tailored reporting**.

```mermaid
flowchart TD
    A[Start: Business Questions<br/>• Does Vein Pack extend life beyond 73 yrs?<br/>• Is Artery Pack different from Vein?<br/>• Do packs affect iron levels?] --> B[Load & Inspect Data<br/>familiar_lifespan.csv<br/>familiar_iron.csv]
    B --> C[Exploratory Analysis<br/>head, describe, value_counts<br/>boxplots / histograms]
    C --> D1[One-Sample t-test<br/>Vein vs μ₀ = 73]
    C --> D2[Two-Sample t-test<br/>Vein vs Artery]
    C --> D3[Chi-Square Test<br/>Pack × Iron Level]
    D1 --> E{ p &lt; 0.05 ? }
    D2 --> E
    D3 --> E
    E -->|Yes| F[Compute Effect Sizes<br/>Cohen's d, Cramer's V<br/>95 % Confidence Intervals]
    E -->|No / Borderline| G[Check Assumptions<br/>Normality Shapiro-Wilk<br/>Equal variance Levene<br/>Retrospective power]
    F --> H[Visualize for Stakeholders<br/>Annotated boxplots<br/>Stacked bar for iron]
    G --> H
    H --> I[Bootstrap / Simulation<br/>Resample means &amp; differences<br/>Sensitivity to α or N]
    I --> J[Prepare Audience-Tailored Report<br/>• Executives: headline + ROI / risk<br/>• Technical: methods + diagnostics<br/>• Mixed: layered sections + appendix]
    J --> K[Decision &amp; Next Steps<br/>Marketing claims, pack recommendations,<br/>further data collection]
    style E fill:#fff3cd,stroke:#856404
    style J fill:#e6f3ff,stroke:#0066cc
    style I fill:#e8f5e9,stroke:#2e7d32
```

**Key takeaway (from the supplied PDFs):** The final reporting box is critical.  
A C-level reader has little time and moderate data literacy; a data-science peer wants full methods and diagnostics.  
Structure the deliverable so each audience can “swoop in” for the level of detail they need (Introduction → Body → Appendix).


## Audience Considerations (from the provided PDFs)

Before you write a single line of the final report, answer three questions about the people who will read it:

1. **Data Literacy** (Jočys – “What to Consider When Considering the Audience”)  
   - Highly data-literate readers (engineers, data scientists, some psychologists) can handle scatterplots, boxplots, confidence intervals and linear models.  
   - Less data-literate readers (literature, fine arts, many executives) need simpler bars, lines and dots; avoid jargon or explain it with friendly metaphors (“a friendly robot sipping warm oil”).

2. **Subject Knowledge & Audience Type** (McMurrey – “Audience and Situation Analysis”)  
   - **Experts / Technicians** – know the product theory or operations; want technical depth.  
   - **Executives** – make business / marketing decisions; have little technical knowledge; need the “so-what” and risk.  
   - **Nonspecialists** – potential subscribers or board members; want clear practical implications.  
   - Mixed audiences are common → use clear headings and section introductions so each group can skip what they do not need.

3. **Report Structure for Multiple Audiences** (paper-structure.pdf)  
   - **Primary collaborator / client** – reads Introduction + Conclusion, skims Body.  
   - **Executive** – only the headlines in Introduction / Conclusion.  
   - **Technical supervisor** – Body + Appendix for quality control.  
   - Therefore organise: most important findings first, put detailed diagnostics in an Appendix, and leave “signposts” everywhere.

**For Familiar specifically:** Marketing & product leadership are the primary audience (executive + some subject knowledge about blood-transfusion packs).  
Data-science peers and the medical-advisory board form secondary audiences.  
Consequently the notebooks and the 1-page summary emphasise:

- Clear verbal conclusions (“Vein Pack subscribers live significantly longer than 73 years”).  
- Visuals that work for both technical and non-technical readers.  
- A short “Business Implications” paragraph after every statistical result.


## 1. Setup – Import libraries & load data


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_1samp, ttest_ind, chi2_contingency, shapiro, levene
import matplotlib.pyplot as plt
import seaborn as sns

# Optional richer output (graceful fallback)
try:
    from statsmodels.stats.weightstats import ttest_ind as sm_ttest_ind
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)

lifespans = pd.read_csv("familiar_lifespan.csv")
iron = pd.read_csv("familiar_iron.csv")

print("=== Lifespans (first 5 rows) ===")
print(lifespans.head())
print("\nShape:", lifespans.shape)
print("\nPack counts:\n", lifespans.pack.value_counts())

print("\n=== Iron (first 5 rows) ===")
print(iron.head())
print("\nShape:", iron.shape)
print("\nPack counts:\n", iron.pack.value_counts())


## 2. Vein Pack – Does it beat the 73-year benchmark?

**Null:** average lifespan of Vein Pack subscribers = 73 years.  
**Alternative:** average lifespan ≠ 73 years.  
α = 0.05.


In [ ]:
vein_pack_lifespans = lifespans.loc[lifespans.pack == "vein", "lifespan"]

vein_mean = np.mean(vein_pack_lifespans)
print(f"Vein Pack mean lifespan: {vein_mean:.3f} years")
print(f"Is it longer than 73?  {'Yes' if vein_mean > 73 else 'No'}")

tstat, pval = ttest_1samp(vein_pack_lifespans, 73)
print(f"\nOne-sample t-test vs 73:")
print(f"  t-statistic = {tstat:.4f}")
print(f"  p-value     = {pval:.6e}")
print(f"  Significant at α=0.05?  {'Yes – reject H0' if pval < 0.05 else 'No'}")

# Quick CI for the mean (using t critical)
n_v = len(vein_pack_lifespans)
se_v = np.std(vein_pack_lifespans, ddof=1) / np.sqrt(n_v)
t_crit = stats.t.ppf(0.975, df=n_v-1)
ci_v = (vein_mean - t_crit * se_v, vein_mean + t_crit * se_v)
print(f"  95% CI for Vein mean: [{ci_v[0]:.2f}, {ci_v[1]:.2f}]")


## 3. Upselling – Vein vs. Artery Pack lifespans

**Null:** μ_Vein = μ_Artery.  
**Alternative:** μ_Vein ≠ μ_Artery.


In [ ]:
artery_pack_lifespans = lifespans.loc[lifespans.pack == "artery", "lifespan"]

artery_mean = np.mean(artery_pack_lifespans)
print(f"Artery Pack mean lifespan: {artery_mean:.3f} years")
print(f"Vein mean - Artery mean   : {vein_mean - artery_mean:.3f} years")

tstat2, pval2 = ttest_ind(vein_pack_lifespans, artery_pack_lifespans)
print(f"\nTwo-sample t-test (Vein vs Artery):")
print(f"  t-statistic = {tstat2:.4f}")
print(f"  p-value     = {pval2:.6f}")
print(f"  Significant at α=0.05?  {'Yes – reject H0' if pval2 < 0.05 else 'No (borderline)'}")


## 4. Side Effects – Iron levels by pack

**Null:** pack type and iron category are independent.  
**Alternative:** there is an association.


In [ ]:
print("Iron data head:")
print(iron.head())

Xtab = pd.crosstab(iron.pack, iron.iron)
print("\nContingency table (counts):")
print(Xtab)

# Proportions (row-wise) for easier interpretation
print("\nRow-wise proportions:")
print(Xtab.div(Xtab.sum(axis=1), axis=0).round(3))

chi2, pval_chi, dof, expected = chi2_contingency(Xtab)
print(f"\nChi-square test:")
print(f"  χ² = {chi2:.3f}, df = {dof}")
print(f"  p-value = {pval_chi:.6e}")
print(f"  Significant at α=0.05?  {'Yes – reject H0' if pval_chi < 0.05 else 'No'}")
print("\nExpected counts under independence:")
print(pd.DataFrame(expected, index=Xtab.index, columns=Xtab.columns).round(1))


## 5. Effect sizes, confidence intervals & assumption checks

### Cohen’s d (standardized mean difference)


In [ ]:
def cohens_d(x, y):
    """Pooled-sd Cohen's d."""
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx-1)*np.var(x, ddof=1) + (ny-1)*np.var(y, ddof=1)) / dof)
    return (np.mean(x) - np.mean(y)) / pooled_std

d = cohens_d(vein_pack_lifespans, artery_pack_lifespans)
print(f"Cohen's d (Vein - Artery) = {d:.3f}")
if abs(d) < 0.2:
    interp = "negligible"
elif abs(d) < 0.5:
    interp = "small"
elif abs(d) < 0.8:
    interp = "medium"
else:
    interp = "large"
print(f"Interpretation: {interp} effect")

# 95 % CI for the difference of means (Welch-style or classic)
n1, n2 = len(vein_pack_lifespans), len(artery_pack_lifespans)
s1, s2 = np.std(vein_pack_lifespans, ddof=1), np.std(artery_pack_lifespans, ddof=1)
se_diff = np.sqrt(s1**2/n1 + s2**2/n2)
# Welch-Satterthwaite df approximation
df_w = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
t_crit_w = stats.t.ppf(0.975, df=df_w)
diff = vein_mean - artery_mean
ci_diff = (diff - t_crit_w * se_diff, diff + t_crit_w * se_diff)
print(f"\n95% CI for (Vein mean - Artery mean): [{ci_diff[0]:.3f}, {ci_diff[1]:.3f}]")
print("(Interval includes 0 → consistent with non-significant two-sample test)")


### Assumption diagnostics


In [ ]:
print("Shapiro-Wilk normality tests (H0: data are normal):")
sw_v, p_v = shapiro(vein_pack_lifespans)
sw_a, p_a = shapiro(artery_pack_lifespans)
print(f"  Vein   : W = {sw_v:.4f}, p = {p_v:.4f}  → {'OK' if p_v > 0.05 else 'possible departure'}")
print(f"  Artery : W = {sw_a:.4f}, p = {p_a:.4f}  → {'OK' if p_a > 0.05 else 'possible departure'}")

print("\nLevene test for equal variances (H0: equal variances):")
lev_stat, lev_p = levene(vein_pack_lifespans, artery_pack_lifespans)
print(f"  statistic = {lev_stat:.4f}, p = {lev_p:.4f}  → {'equal var OK' if lev_p > 0.05 else 'variances differ'}")

# Cramér's V
n_total = Xtab.values.sum()
cramers_v = np.sqrt(chi2 / (n_total * (min(Xtab.shape) - 1)))
print(f"\nCramér's V (iron association) = {cramers_v:.3f}")
if cramers_v < 0.1:
    v_interp = "negligible"
elif cramers_v < 0.3:
    v_interp = "small"
elif cramers_v < 0.5:
    v_interp = "medium"
else:
    v_interp = "large"
print(f"Interpretation: {v_interp} association")


## 6. Visualizations

Boxplots for the technical audience; a clean annotated bar chart that executives can read at a glance.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplots
sns.boxplot(data=lifespans, x="pack", y="lifespan", hue="pack", ax=axes[0],
            order=["vein", "artery"], palette=["#4c72b0", "#dd8452"], legend=False)
axes[0].axhline(73, color="red", ls="--", lw=1.5, label="Population benchmark (73 yr)")
axes[0].set_title("Lifespan by Pack\n(with 73-year reference line)")
axes[0].set_ylabel("Lifespan (years)")
axes[0].legend(loc="lower right")

# Strip / swarm for extra detail
sns.stripplot(data=lifespans, x="pack", y="lifespan", ax=axes[0],
              order=["vein", "artery"], color="black", alpha=0.4, size=4)

# Iron proportions – stacked bar
props = Xtab.div(Xtab.sum(axis=1), axis=0)
props = props[["low", "normal", "high"]]  # consistent order
props.plot(kind="bar", stacked=True, ax=axes[1],
           color=["#d62728", "#2ca02c", "#1f77b4"], rot=0)
axes[1].set_title("Iron Level Distribution by Pack")
axes[1].set_ylabel("Proportion of subscribers")
axes[1].set_xlabel("Pack")
axes[1].legend(title="Iron", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig("familiar_visuals.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved as familiar_visuals.png")


## 7. Simulation & Sensitivity Section

### 7.1 Bootstrap distribution of the mean difference


In [ ]:
np.random.seed(42)
n_boot = 2000
vein_arr = vein_pack_lifespans.values
artery_arr = artery_pack_lifespans.values

boot_diffs = np.empty(n_boot)
for i in range(n_boot):
    b_vein = np.random.choice(vein_arr, size=len(vein_arr), replace=True)
    b_artery = np.random.choice(artery_arr, size=len(artery_arr), replace=True)
    boot_diffs[i] = b_vein.mean() - b_artery.mean()

ci_boot = np.percentile(boot_diffs, [2.5, 97.5])
pct_positive = (boot_diffs > 0).mean() * 100

print(f"Bootstrap 95% CI for (Vein - Artery): [{ci_boot[0]:.3f}, {ci_boot[1]:.3f}]")
print(f"% of bootstrap samples with Vein > Artery: {pct_positive:.1f}%")

plt.figure(figsize=(8, 4))
plt.hist(boot_diffs, bins=40, color="#4c72b0", edgecolor="white", alpha=0.85)
plt.axvline(0, color="red", ls="--", label="Zero difference")
plt.axvline(ci_boot[0], color="green", ls=":", label="2.5 / 97.5 percentiles")
plt.axvline(ci_boot[1], color="green", ls=":")
plt.title("Bootstrap distribution of mean difference (Vein − Artery)")
plt.xlabel("Mean difference (years)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


### 7.2 Sensitivity of the one-sample test to the null value μ₀


In [ ]:
print("One-sample t-test p-values for different null values:")
for mu0 in [72, 73, 74, 75, 76]:
    _, p = ttest_1samp(vein_pack_lifespans, mu0)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  μ₀ = {mu0:4.1f}  →  p = {p:.4e}  {sig}")


### 7.3 What-if: drastically smaller sample size


In [ ]:
np.random.seed(7)
# draw 8 observations from each pack
small_vein = np.random.choice(vein_arr, 8, replace=False)
small_artery = np.random.choice(artery_arr, 8, replace=False)
t_small, p_small = ttest_ind(small_vein, small_artery)
print(f"With only n=8 per group:")
print(f"  means: Vein={small_vein.mean():.2f}, Artery={small_artery.mean():.2f}")
print(f"  t = {t_small:.3f}, p = {p_small:.4f}")
print("  → Significance often disappears; adequate sample size matters.")


## 8. Alternate Code Paths

### 8.1 Manual one-sample t-statistic + survival function


In [ ]:
# Classic formula
xbar = vein_mean
s = np.std(vein_pack_lifespans, ddof=1)
n = len(vein_pack_lifespans)
t_manual = (xbar - 73) / (s / np.sqrt(n))
# two-sided p-value from t distribution
p_manual = 2 * stats.t.sf(np.abs(t_manual), df=n-1)
print(f"Manual t = {t_manual:.4f}")
print(f"Manual p = {p_manual:.6e}")
print(f"Matches scipy? {np.isclose(t_manual, tstat) and np.isclose(p_manual, pval)}")


### 8.2 Contingency table via groupby + unstack


In [ ]:
Xtab_alt = iron.groupby(["pack", "iron"]).size().unstack(fill_value=0)
# re-order columns for readability
Xtab_alt = Xtab_alt[["low", "normal", "high"]]
print(Xtab_alt)
print("\nIdentical to crosstab?", Xtab_alt.equals(Xtab[["low", "normal", "high"]]))


### 8.3 statsmodels two-sample t-test (if available)


In [ ]:
if HAS_STATSMODELS:
    t_sm, p_sm, df_sm = sm_ttest_ind(vein_pack_lifespans, artery_pack_lifespans, usevar="pooled")
    print(f"statsmodels t = {t_sm:.4f}, p = {p_sm:.6f}, df = {df_sm:.1f}")
else:
    print("statsmodels not installed – skipping alternate implementation.")


## 9. Business Implications & Mini Report Draft

**Problem Statement**  
Familiar needs evidence that its subscription packs improve lifespan relative to the general population and that the two packs (Vein vs Artery) differ in side-effect profiles, so marketing claims can be substantiated and product positioning refined.

**Key Findings**
- Vein Pack mean lifespan ≈ 76.2 years – **highly significantly** longer than the 73-year benchmark (p ≈ 6 × 10⁻⁷).  
- Artery Pack mean ≈ 74.9 years; the 1.3-year difference versus Vein is **borderline** (p ≈ 0.056) and the 95 % CI for the difference includes zero. Cohen’s d ≈ 0.62 (medium).  
- Strong association between pack and iron level (χ² p ≈ 10⁻²⁴, Cramér’s V ≈ 0.55 – large). Vein subscribers are far more likely to have **low** iron; Artery subscribers are far more likely to have **high** iron.

**Recommendation**
- Marketing can confidently claim a longevity benefit for the Vein Pack.  
- The Artery Pack does **not** yet show a statistically clear longevity advantage over Vein; larger samples or a different endpoint may be required.  
- Side-effect counselling is essential: Vein → monitor for iron deficiency; Artery → monitor for iron overload.  This differentiation can be turned into a product-feature story (“choose the pack that matches your iron profile”).

**Limitations & Next Steps**
- Lifespan sample is modest (n = 20 per arm); a prospective study with larger N and longer follow-up would strengthen causal claims.  
- Iron data are categorical and cross-sectional; longitudinal haemoglobin / ferritin measurements would be more clinically actionable.  
- No demographic covariates were supplied – residual confounding cannot be ruled out.


## End of Solution Notebook

You now have a complete, audience-aware analysis pipeline that can be dropped into Familiar’s next product-review meeting.  
Re-run the bootstrap cell with different seeds or change the sub-sample size in section 7.3 to explore sensitivity further.
